In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')



In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
# Task 1: Write your code here:
import os
file_ = os.path.join(path,"Q1_data.csv")
df = pd.read_csv(file_)

In [ ]:
# Task 2: Write your code here:
print(df.head())

In [ ]:
# Task 3: Write your code here:
print(df.info())

In [ ]:
# Task 4: Write your code here:
print(df.describe())

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,10))
plt.hist(df["Delivery_Time"],color="red")
plt.title("Target Distribution")
plt.show()

In [ ]:
# Task 1: Write your code here:
df.drop(columns=["Order_ID"],inplace=True)

In [ ]:
# Task 2: Write your code here:
df["Weather"] = df["Weather"].fillna(df["Weather"].mode()[0])
df["Traffic_Level"] = df["Traffic_Level"].fillna(df["Traffic_Level"].mode()[0])
df["Time_of_Day"] = df["Time_of_Day"].fillna(df["Time_of_Day"].mode()[0])
df["Courier_Experience_yrs"] = df["Courier_Experience_yrs"].fillna(df["Courier_Experience_yrs"].median())
df.dropna(subset=["Delivery_Time"],inplace=True)

In [ ]:
# Task 3: Write your code here:
df.drop_duplicates(inplace=True)

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import OneHotEncoder #import OneHotEncoder
categories = ["Weather","Traffic_Level","Time_of_Day","Vehicle_Type"]
df_test = df[categories].copy()
for col in categories:

  onehot_encoder = OneHotEncoder(sparse_output=False) # Instantiate OneHotEncoder
  df_encoded = onehot_encoder.fit_transform(df_test) # Apply fit_transform to the copied
  df[col] = df_encoded

print('\nData after encoding:\n', df) #show after encoding


In [ ]:
# Task 5: Write your code here:
# Scale features - fit on train, transform both
#df.drop(columns=["Time"],inplace=True)
scaler = StandardScaler()
columns = df.columns.drop("Delivery_Time")
df[columns] = scaler.fit_transform(df[columns])




In [ ]:
# Task 6: Write your code here:
print(df["Delivery_Time"].value_counts())

In [ ]:
# Task 1: Write your code here:
feature = df.drop(columns=["Delivery_Time"])
target = df["Delivery_Time"].copy()

In [ ]:
# Task 2,3,4,5: Write your code here:
skf = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
rf = RandomForestRegressor(random_state=42)

error = []
y_pred = ""
for train_idx, test_idx  in skf.split(feature,target):
  X_train, X_test = feature.iloc[train_idx], feature.iloc[test_idx]
  y_train, y_test = target.iloc[train_idx], target.iloc[test_idx]

  # Train
  rf.fit(X_train,y_train)

  y_pred = rf.predict(X_test)

  # Calculate evaluation metrics
  mae = mean_absolute_error(y_test, y_pred)

  # Store results
  error.append(mae)
print(f"Average score:- {np.mean(error):.4f}")

In [ ]:
# Task 1: Write your code here:
importance = rf.feature_importances_

# Create a 1x3 plot
plt.figure(figsize=(10,10))
features = feature.columns
plt.barh(features[np.argsort(importance)],width=20)
plt.title("Importance")


plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(10,10))
plt.hist(y_pred,color="blue")
plt.show()

In [ ]:
!pip install catboost

In [ ]:
# Task Bonus: Write your code here:
# TODO: Define models with hyperparameters of your choice
from catboost import CatBoostRegressor
models = {
  "RF": RandomForestRegressor(random_state=42),
  "CatBoost": CatBoostRegressor(verbose=0)
}


# Storage for results
all_results = {}

for name in models:
  all_results[name] = {'mae': []}

skf = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
rf = RandomForestRegressor(random_state=42)

error = []
y_pred = ""
for train_idx, test_idx  in skf.split(feature,target):
  X_train, X_test = feature.iloc[train_idx], feature.iloc[test_idx]
  y_train, y_test = target.iloc[train_idx], target.iloc[test_idx]

  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Calculate metrics
    mae = mean_absolute_error(y_test, y_pred)

    # Store results
    all_results[model_name]["mae"].append(mae)


for model_name in all_results:
  print(f"\n{model_name}:")
  print(f"  MAE:  {np.mean(all_results[model_name]['mae']):.4f}")